In [14]:
import sys
import torch
sys.path.append('/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/domain_experiment/satclip/satclip')
from model import VisionTransformer# or wherever it's defined
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
import torchvision
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import os
import time
from PIL import Image



In [2]:
checkpoint = torch.load("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/domain_experiment/satclip/satclip/satclip_logs/satclip/satclip/checkpoints/last-v1.ckpt", map_location="cpu")
state_dict = checkpoint["state_dict"] 
vit_state_dict = {
    k.replace("model.visual.", ""): v
    for k, v in state_dict.items()
    if k.startswith("model.visual.")
}


In [3]:
# Define the VisionTransformer model with the same architecture as used in the checkpoint
vit_model = VisionTransformer(
    input_resolution=640,
    patch_size=16,
    width=768,
    layers=12,
    heads=12,
    output_dim=768,
    in_channels=3,
)

vit_model.load_state_dict(vit_state_dict, strict=True)



<All keys matched successfully>

In [4]:
class ViTBackboneForDetection(nn.Module):
    def __init__(self, vit):
        super().__init__()
        self.vit = vit
        self.out_channels = 768  

    def forward(self, x):
        _, patch_tokens = self.vit(x)
        B, N, C = patch_tokens.shape  # [B, 1600, 768]
        # print('patch_tokens shape', patch_tokens.shape)
        H = W = int(N ** 0.5)         # H = W = 40 for 640x640
        features = patch_tokens.permute(0, 2, 1).reshape(B, C, H, W)  # [B, 768, 40, 40]
        return {"0": features}
    
vit_backbone = ViTBackboneForDetection(vit_model)

transform = GeneralizedRCNNTransform(
    min_size=640,
    max_size=640,
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225]
)


anchor_generator = AnchorGenerator(
    sizes=((8, 16, 32, 64),),
    aspect_ratios=((0.5, 1.0, 2.0),)
)

# Define ROI pooling
roi_pooler = torchvision.ops.MultiScaleRoIAlign(
    featmap_names=["0"],
    output_size=7,
    sampling_ratio=2
)

In [5]:
model = FasterRCNN(
    backbone=vit_backbone,
    num_classes=4,  
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler,
)
model.transform = transform
model.roi_heads.box_detections_per_img = 300  # increase max detections per image
model.roi_heads.nms_thresh = 0.3  # adjust NMS
model

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640,), max_size=640, mode='bilinear')
  )
  (backbone): ViTBackboneForDetection(
    (vit): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): Sequential(
          (0): ResidualAttentionBlock(
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): QuickGELU()
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ln_2): 

In [6]:
if 0:
    model.eval()
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Create dummy input
    input_tensor = torch.randn(3, 640, 640).to(device)

    # Wrap as list and run model
    with torch.no_grad():
        output = model([input_tensor])  
    # Output
    print("Number of detections:", len(output[0]['boxes']))
    print("Boxes shape:", output[0]['boxes'].shape)
    print("Labels shape:", output[0]['labels'].shape)
    print("Scores shape:", output[0]['scores'].shape)

In [7]:
image_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/images"
label_dir_yolo = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/labels"
label_dir_voc = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/label_aa_voc"


In [8]:
transform=transforms.Compose([
    transforms.ToTensor()])


# Set device
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

class TIFRCNNDataset(Dataset):
    def __init__(self, image_dir, label_dir, transforms=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transforms = transforms
        # Include all .tif images regardless of label presence
        self.image_filenames = [f for f in os.listdir(image_dir) if f.endswith('.tif')]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        label_path = os.path.join(self.label_dir, os.path.splitext(img_name)[0] + ".txt")

        boxes = []
        labels = []

        # If label file exists, read boxes and labels
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    if line.strip():
                        try:
                            cls, x1, y1, x2, y2 = map(float, line.strip().split())
                            boxes.append([x1, y1, x2, y2])
                            labels.append(int(cls) + 1)  # background=0
                        except:
                            pass

        # Convert to tensors, or empty tensors if no labels
        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))

In [9]:
dataset= TIFRCNNDataset(image_dir, label_dir_voc, transforms=transform)
dataloader = DataLoader(dataset, batch_size=24,pin_memory=True, shuffle=True, collate_fn=collate_fn)


In [17]:
model.load_state_dict(torch.load(
    "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/domain_experiment/notebooks/faster_rcnn_epoch_10.pth",
    map_location="cpu"
), strict=True)
model.to(device)
model.eval()

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640,), max_size=640, mode='bilinear')
  )
  (backbone): ViTBackboneForDetection(
    (vit): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): Sequential(
          (0): ResidualAttentionBlock(
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): QuickGELU()
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ln_2): 

In [11]:
model.roi_heads.nms_thresh = 0.50

In [ ]:
if 0:



    model.to(device)
    model.train()

    # Optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=1e-4)

    # Training loop
    num_epochs = 20
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        start_time = time.time()

        # Use tqdm for the dataloader loop
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")
        for images, targets in pbar:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Forward pass and loss computation
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            # Backward pass
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()

            # Update tqdm description
            pbar.set_postfix(loss=losses.item())

        end_time = time.time()
        print(f"[Epoch {epoch+1}/{num_epochs}] Total Loss: {epoch_loss:.4f} | Time: {end_time - start_time:.2f}s")
        # Save model checkpoint
    checkpoint_path = os.path.join(f"faster_rcnn_epoch_{epoch+1}.pth")
    torch.save(model.state_dict(), checkpoint_path)


In [18]:
# -------------------------
# Inference Loop
# -------------------------
# model.roi_heads.nms_thresh = 0.33
model.eval()
output_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/lucknow_airshed/output_1"
#delete output_dir
if os.path.exists(output_dir):
    import shutil
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)
CONF_THRESH=0.05
with torch.no_grad():
    for filename in tqdm(os.listdir(image_dir)):
        if not filename.endswith((".jpg", ".png", ".tif")):
            continue

        # Load and preprocess image
        image_path = os.path.join(image_dir, filename)
        image = Image.open(image_path).convert("RGB")
        image_tensor = transforms.ToTensor()(image)  # Only ToTensor here
        image_tensor = image_tensor.to(device)

        # Pass through model
        outputs = model([image_tensor])[0]

        boxes = outputs["boxes"]
        labels = outputs["labels"]
        scores = outputs["scores"]

        # Skip if no detections
        if len(boxes) == 0:
            continue

        # Filter and format predictions
        pred_lines = []
        for label, score, box in zip(labels, scores, boxes):
            if score < CONF_THRESH:
                continue
            x1, y1, x2, y2 = box.tolist()
            line = f"{label.item() - 1} {score.item():.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}"
            pred_lines.append(line)

        # Save predictions
        out_file = os.path.join(output_dir, os.path.splitext(filename)[0] + ".txt")
        with open(out_file, "w") as f:
            f.write("\n".join(pred_lines))

  0%|          | 0/598 [00:00<?, ?it/s]

100%|██████████| 598/598 [01:07<00:00,  8.85it/s]


In [ ]:
model.roi_heads.nms_thresh = 0.50

In [ ]:
# # Suggested improvements and debug instrumentation for ViT+FasterRCNN model training

# import torch
# import torchvision
# from torchvision.models.detection import FasterRCNN
# from torchvision.models.detection.rpn import AnchorGenerator
# from torchvision.ops import MultiScaleRoIAlign
# from torchvision.models.detection.transform import GeneralizedRCNNTransform
# from torchvision.transforms import functional as F
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# from PIL import Image
# import numpy as np

# # -------------------- Transform (for Dataset) --------------------
# transform = A.Compose([
#     A.Resize(640, 640),
#     A.HorizontalFlip(p=0.5),
#     A.RandomBrightnessContrast(p=0.5),
#     A.Normalize(mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]),
#     ToTensorV2()
# ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# # -------------------- ViT Backbone --------------------
# class ViTBackboneForDetection(torch.nn.Module):
#     def __init__(self, vit):
#         super().__init__()
#         self.vit = vit
#         self.out_channels = 768

#     def forward(self, x):
#         _, patch_tokens = self.vit(x)
#         B, N, C = patch_tokens.shape
#         H = W = int(N ** 0.5)
#         features = patch_tokens.permute(0, 2, 1).reshape(B, C, H, W)
#         return {"0": features}

# # -------------------- Anchor Generator --------------------
# # Adjust for small 32x32 objects
# anchor_generator = AnchorGenerator(
#     sizes=((8, 16, 32, 64),),
#     aspect_ratios=((0.5, 1.0, 2.0),)
# )

# # -------------------- RoI Pooling --------------------
# roi_pooler = MultiScaleRoIAlign(
#     featmap_names=['0'],
#     output_size=7,
#     sampling_ratio=2
# )

# # -------------------- Transform Override --------------------
# transform_override = GeneralizedRCNNTransform(
#     min_size=640,
#     max_size=640,
#     image_mean=[0.485, 0.456, 0.406],
#     image_std=[0.229, 0.224, 0.225]
# )

# # -------------------- Model Assembly --------------------
# model = FasterRCNN(
#     backbone=ViTBackboneForDetection(vit_model),  # your pretrained SatCLIP ViT
#     num_classes=4,  # including background
#     rpn_anchor_generator=anchor_generator,
#     box_roi_pool=roi_pooler
# )
# model.transform = transform_override
# model.roi_heads.box_detections_per_img = 300  # increase max detections per image
# model.roi_heads.nms_thresh = 0.3  # adjust NMS

# # -------------------- Optimizer --------------------
# params = [p for p in model.parameters() if p.requires_grad]
# optimizer = torch.optim.Adam(params, lr=1e-4)

# # -------------------- Debug: Visualize One Sample --------------------
# def visualize_sample(img, target):
#     import matplotlib.pyplot as plt
#     from matplotlib.patches import Rectangle

#     img = img.permute(1, 2, 0).cpu().numpy()
#     plt.imshow(img)
#     for box in target['boxes']:
#         x1, y1, x2, y2 = box.tolist()
#         rect = Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor='red', facecolor='none')
#         plt.gca().add_patch(rect)
#     plt.title("Sanity check: input + GT boxes")
#     plt.show()

# # Use this in DataLoader loop to debug a few samples visually
